# Headout EN → FR  ·  NLLB-200-1.3B + QLoRA

**Model** : `facebook/nllb-200-distilled-1.3B` (1.3B params, purpose-built for translation)  
**Method** : QLoRA — 4-bit NF4 base + LoRA adapters on attention layers (~20M trainable params)  
**Data**  : 100K stratified pairs, pre-parsed locally and uploaded as JSON  
**Target** : NVIDIA L4 24 GB · estimated **~1.5 hrs** total training

### Key differences from previous MarianMT approach
- No vocabulary extension — NLLB tokenizer handles all placeholder characters natively
- Language routing via `eng_Latn` → `fra_Latn` tokens (not a separate tokenizer)
- `predict_with_generate=False` during training — tracks loss only, BLEU evaluated once after
- `paged_adamw_8bit` optimizer from bitsandbytes — keeps optimizer states in 8-bit

## 1 · GPU Check

In [ ]:
import os, subprocess, torch

r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'nvidia-smi not found')

assert torch.cuda.is_available(), 'No GPU — this notebook requires a CUDA GPU'
DEVICE = 'cuda'
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
assert torch.cuda.is_bf16_supported(), (
    'BF16 not supported. Change bnb_4bit_compute_dtype and bf16= to torch.float16 / fp16=True'
)
print('BF16 : supported ✓')

# Reduces fragmentation — helps avoid OOM on 22 GB effective VRAM
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# TF32 — free speedup on Ada GPUs (L4), no quality loss
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print('TF32 : enabled ✓')

## 2 · Install Dependencies

In [ ]:
!pip install -q \
    'transformers>=4.40.0' \
    'datasets>=2.19.0' \
    'peft>=0.10.0' \
    'bitsandbytes>=0.43.0' \
    'accelerate>=0.29.0' \
    'sacrebleu>=2.4.0' \
    'sacremoses>=0.1.1' \
    'sentencepiece>=0.1.99' \
    'tensorboard>=2.16.0'
print('Done.')

## 3 · Config

In [ ]:
import math
from pathlib import Path

MODEL_NAME   = 'facebook/nllb-200-distilled-1.3B'
SRC_LANG     = 'eng_Latn'
TGT_LANG     = 'fra_Latn'
MAX_LEN      = 64

# LoRA
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# Training  (L4 24 GB)
# NLLB vocab=256K → logits tensor = batch × 64 × 256K × 2 bytes
# batch=32  → 1.05 GB logits ← fits safely
# batch=128 → 3.91 GB logits ← OOM on 22 GB effective VRAM
TRAIN_BATCH             = 32
GRAD_ACCUM              = 4    # effective batch = 32 × 4 = 128
EVAL_BATCH              = 32
LEARNING_RATE           = 2e-4
WARMUP_STEPS            = 100
NUM_EPOCHS              = 3
EVAL_STEPS              = 200
ES_PATIENCE             = 5

OUT_DIR      = './checkpoints'
ADAPTER_DIR  = './lora-adapter'
BEST_DIR     = f'{OUT_DIR}/best'

train_size      = 90_000
steps_epoch     = math.ceil(train_size / TRAIN_BATCH)   # raw steps
optimizer_steps = math.ceil(steps_epoch / GRAD_ACCUM) * NUM_EPOCHS
print(f'Raw steps/epoch    : {steps_epoch:,}')
print(f'Optimizer updates  : {optimizer_steps:,}  (same as batch=128 direct)')
print(f'Est. time          : ~{optimizer_steps / 60:.0f} min on L4')

## 4 · Upload Pre-Parsed JSON Files

Run `make parse` locally first, then upload `train.json`, `val.json`, `test.json`.
Each file is ~10–25 MB — much faster than uploading the full TMX.

In [ ]:
import os, json
from pathlib import Path

Path('data').mkdir(exist_ok=True)

missing = [s for s in ['train', 'val', 'test'] if not Path(f'data/{s}.json').exists()]
if missing:
    try:
        from google.colab import files
        print(f'Upload {missing} JSON files:')
        uploaded = files.upload()
        for name, content in uploaded.items():
            with open(f'data/{name}', 'wb') as f:
                f.write(content)
            print(f'  data/{name}  ({len(content)/1e6:.1f} MB)')
    except ImportError:
        raise FileNotFoundError(
            f'Missing: {missing}. Upload them via the Workbench file browser '
            f'or gsutil cp gs://YOUR_BUCKET/{{train,val,test}}.json data/'
        )

for split in ['train', 'val', 'test']:
    p = Path(f'data/{split}.json')
    with open(p) as f:
        n = len(json.load(f))
    print(f'data/{split}.json  →  {n:,} pairs  ({p.stat().st_size/1e6:.1f} MB)')

## 5 · Placeholder Normalization

**Simpler than before** — no vocabulary extension.
- XML `<ph>`, `<bpt>`, `<ept>` tags → `[PH_N]`, `[BPT_N]`, `[EPT_N]`
- Bare URLs → `[URL_N]`
- Curly icon keys `{clock}`, `{skip}` etc. kept as-is — NLLB tokenizes them as rare
  subword pieces and learns to copy them through cross-attention
- Side-channel dict maps each token back to its original for restoration

In [ ]:
import re, io

_PH_XML   = re.compile(r'<ph\b[^>]*/></ph>|<ph\b[^>]*>.*?</ph>', re.DOTALL)
_PH_IDX   = re.compile(r'x="(\d+)"')
_BPT      = re.compile(r'<bpt\b[^>]*>.*?</bpt>', re.DOTALL)
_EPT      = re.compile(r'<ept\b[^>]*>.*?</ept>', re.DOTALL)
_TAG_IDX  = re.compile(r'i="(\d+)"')
_MD_LINK  = re.compile(r'\[([^\]]+)\]\s*\((https?://[^)]+)\)')
_BARE_URL = re.compile(r'https?://\S+')
_BROKEN   = re.compile(r'\{([^})]{1,40}?)\)')   # {foo) → {foo}

CURLY_FR_TO_EN = {
    'transferts': 'transfers', 'annulation': 'cancel',
    'horloge': 'clock',        'calendrier': 'calendar',
    'pone': 'phone',           "casques d'écoute": 'headphones',
    'casques': 'headphones',   'écouteurs': 'headphones',
}

def _fix_curly(text):
    text = _BROKEN.sub(lambda m: '{' + m.group(1) + '}', text)
    def _rep(m):
        key = m.group(1).strip().lower()
        return '{' + CURLY_FR_TO_EN.get(key, key) + '}'
    return re.sub(r"\"{([a-zA-ZéàîôùèêâûœçÉÀÎÔÙÈÊÂÛŒÇ _']{1,40}?)}\"",
                  _rep, text)

def normalize(text, sc):
    text = _fix_curly(text)

    ph = {}
    def _r_ph(m):
        mi = _PH_IDX.search(m.group(0))
        idx = mi.group(1) if mi else str(len(ph))
        ph[idx] = m.group(0); return f'[PH_{idx}]'
    text = _PH_XML.sub(_r_ph, text)
    if ph: sc['ph'] = ph

    bpt, ept = {}, {}
    def _r_bpt(m):
        mi = _TAG_IDX.search(m.group(0))
        idx = mi.group(1) if mi else str(len(bpt))
        bpt[idx] = m.group(0); return f'[BPT_{idx}]'
    def _r_ept(m):
        mi = _TAG_IDX.search(m.group(0))
        idx = mi.group(1) if mi else str(len(ept))
        ept[idx] = m.group(0); return f'[EPT_{idx}]'
    text = _BPT.sub(_r_bpt, text)
    text = _EPT.sub(_r_ept, text)
    if bpt: sc['bpt'] = bpt; sc['ept'] = ept

    urls = {}
    ctr  = [0]
    def _r_md(m):
        k = f'URL_{ctr[0]}'; urls[k] = m.group(2); ctr[0] += 1
        return f'[{m.group(1)}]({k})'
    text = _MD_LINK.sub(_r_md, text)
    def _r_url(m):
        idx = len(urls); urls[f'BARE_{idx}'] = m.group(0)
        return f'[URL_{idx}]'
    text = _BARE_URL.sub(_r_url, text)
    if urls: sc['urls'] = urls

    text = re.sub(r'\]\s+\(', '](', text)
    return text

def restore(norm_fr, sc):
    for idx, orig in sc.get('ph',  {}).items(): norm_fr = norm_fr.replace(f'[PH_{idx}]',  orig)
    for idx, orig in sc.get('bpt', {}).items(): norm_fr = norm_fr.replace(f'[BPT_{idx}]', orig)
    for idx, orig in sc.get('ept', {}).items(): norm_fr = norm_fr.replace(f'[EPT_{idx}]', orig)
    for key, url in sc.get('urls', {}).items():
        if key.startswith('BARE_'):
            norm_fr = norm_fr.replace(f'[URL_{key.split("_",1)[1]}]', url)
        else:
            norm_fr = norm_fr.replace(key, url)
    return norm_fr

def validate_ph(norm_en, norm_fr):
    pat = re.compile(r'\[(?:PH|BPT|EPT|URL)_\d+\]')
    return set(pat.findall(norm_en)) == set(pat.findall(norm_fr))

def repair_ph(norm_en, norm_fr):
    pat = re.compile(r'\[(?:PH|BPT|EPT|URL)_\d+\]')
    missing = [t for t in pat.findall(norm_en) if t not in norm_fr]
    if missing: norm_fr = norm_fr.rstrip() + ' ' + ' '.join(missing)
    return norm_fr

# Smoke test
sc = {}
n = normalize('<ph x="0" type="x"><x/></ph>Skip the Line {clock}', sc)
assert '[PH_0]' in n and '{clock}' in n
assert restore(n, sc) == '<ph x="0" type="x"><x/></ph>Skip the Line {clock}'
print('Normalization smoke test passed ✓')

## 6 · Load NLLB Tokenizer

In [ ]:
from transformers import AutoTokenizer

print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.src_lang = SRC_LANG

# Fast tokenizer (TokenizersBackend) doesn't have lang_code_to_id —
# convert_tokens_to_ids works on both fast and slow variants
fra_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
assert fra_token_id != tokenizer.unk_token_id, \
    f'{TGT_LANG} not found in vocab — check the language code'

print(f'Vocab size       : {len(tokenizer):,}')
print(f'fra_Latn token id: {fra_token_id}')

# Show how placeholder strings tokenize (subwords, no extension needed)
for s in ['[PH_0]', '[BPT_1]', '[URL_0]', '{clock}', '{skip}']:
    ids = tokenizer.encode(s, add_special_tokens=False)
    print(f'  {s!r:12} → {len(ids)} token(s): {ids}')

## 7 · Load Model in 4-bit + Apply LoRA

In [ ]:
from transformers import AutoModelForSeq2SeqLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f'Loading {MODEL_NAME} in 4-bit ...')
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
)
print(f'GPU memory after load: {torch.cuda.memory_allocated(0)/1e9:.2f} GB')

# Gradient checkpointing ON — re-computes activations to save VRAM,
# needed to fit batch=128 within 22 GB effective VRAM on L4
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj'],
    lora_dropout=LORA_DROPOUT,
    bias='none',
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
print(f'GPU memory after LoRA: {torch.cuda.memory_allocated(0)/1e9:.2f} GB')

## 8 · Tokenize Datasets

In [ ]:
from datasets import Dataset as HFDataset
from transformers import DataCollatorForSeq2Seq

def load_split(path):
    with open(path) as f:
        return HFDataset.from_list(json.load(f))

def preprocess(batch):
    # tokenizer.src_lang already set to eng_Latn above
    return tokenizer(
        text=batch['en'],
        text_target=batch['fr'],
        max_length=MAX_LEN,
        truncation=True,
        padding=False,
    )

print('Loading and tokenizing ...')
train_ds = load_split('data/train.json').map(
    preprocess, batched=True, remove_columns=['en', 'fr'], num_proc=2
)
val_ds = load_split('data/val.json').map(
    preprocess, batched=True, remove_columns=['en', 'fr'], num_proc=2
)
# Test set loaded raw — inference loop handles tokenization
with open('data/test.json') as f:
    test_records = json.load(f)

collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, padding=True, pad_to_multiple_of=8
)
print(f'train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_records):,}')
print('Tokenization complete ✓')

## 9 · Fine-Tune

`predict_with_generate=False` — only cross-entropy loss is computed during eval steps.  
Full BLEU / chrF++ / placeholder-parity evaluation runs once in cell 11 after training.

In [ ]:
import time
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
)

class StepLogger(TrainerCallback):
    def __init__(self): self._t0 = time.time()
    def on_log(self, args, state, control, logs=None, **kw):
        if logs:
            loss = logs.get('loss') or logs.get('eval_loss', '?')
            print(f'[{time.time()-self._t0:6.0f}s | step {state.global_step:5d}] loss={loss}  {logs}', flush=True)

steps_epoch     = math.ceil(len(train_ds) / TRAIN_BATCH)
optimizer_steps = math.ceil(steps_epoch / GRAD_ACCUM) * NUM_EPOCHS
print(f'Raw steps/epoch  : {steps_epoch:,}')
print(f'Optimizer updates: {optimizer_steps:,}')
print(f'Est. time        : ~{optimizer_steps / 60:.0f} min on L4\n')

training_args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,

    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,   # effective batch = 32 × 4 = 128

    optim='paged_adamw_8bit',
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    max_grad_norm=1.0,

    bf16=True,
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,

    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    save_steps=EVAL_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    predict_with_generate=False,

    logging_steps=50,
    report_to='tensorboard',
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=collator,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=ES_PATIENCE),
        StepLogger(),
    ],
)

print('Starting training ...')
trainer.train()

## 10 · Save LoRA Adapter

In [ ]:
# Save just the LoRA adapter weights (~80 MB) — much smaller than a full checkpoint
Path(ADAPTER_DIR).mkdir(exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'LoRA adapter saved to {ADAPTER_DIR}/')
adapter_size = sum(p.stat().st_size for p in Path(ADAPTER_DIR).rglob('*') if p.is_file())
print(f'Adapter size: {adapter_size/1e6:.1f} MB')

## 11 · Evaluate on Test Set  (BLEU · chrF++ · Placeholder Parity)

In [ ]:
import sacrebleu as scb

# Reload the best adapter for clean eval
# (If you just finished training, 'model' already holds the best weights
#  due to load_best_model_at_end=True — you can skip the reload block.)
# from peft import PeftModel
# base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map='auto')
# model = PeftModel.from_pretrained(base, ADAPTER_DIR)

model.eval()
EVAL_BS = 32   # 32 × 4 beams = 128 concurrent sequences — safe for 24 GB

en_texts = [r['en'] for r in test_records]
fr_refs  = [r['fr'] for r in test_records]
predictions = []

print(f'Translating {len(en_texts):,} test pairs ...')
t_start = time.time()
for i in range(0, len(en_texts), EVAL_BS):
    batch = en_texts[i:i + EVAL_BS]
    inputs = tokenizer(
        batch, return_tensors='pt',
        max_length=MAX_LEN, truncation=True, padding=True,
    ).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            forced_bos_token_id=fra_token_id,
            num_beams=4,
            length_penalty=0.6,
            max_new_tokens=MAX_LEN,
        )
    predictions.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    if i % (EVAL_BS * 20) == 0:
        done = min(i + EVAL_BS, len(en_texts))
        elapsed = time.time() - t_start
        print(f'  {done:,}/{len(en_texts):,}  ({elapsed:.0f}s)', flush=True)

print(f'Generation done in {time.time()-t_start:.0f}s')

# BLEU
bleu = scb.corpus_bleu(predictions, [fr_refs])
# chrF++
chrf = scb.corpus_chrf(predictions, [fr_refs], word_order=2)

# Placeholder parity
ph_pat  = re.compile(r'\[(?:PH|BPT|EPT|URL)_\d+\]')
curl_pat = re.compile(r'\{\w+\}')
has_ph   = [(i, e) for i, e in enumerate(en_texts) if ph_pat.search(e)]
ph_ok    = sum(1 for i, e in has_ph if validate_ph(e, predictions[i]))
ph_acc   = ph_ok / len(has_ph) * 100 if has_ph else 100.0

has_curl  = [(i, e) for i, e in enumerate(en_texts) if curl_pat.search(e)]
curl_ok   = sum(
    1 for i, e in has_curl
    if all(t in predictions[i] for t in curl_pat.findall(e))
)
curl_acc  = curl_ok / len(has_curl) * 100 if has_curl else 100.0

print('\n' + '='*52)
print(f'BLEU           : {bleu.score:.2f}   (target ≥ 48)')
print(f'chrF++         : {chrf.score:.2f}   (target ≥ 65)')
print(f'XML PH parity  : {ph_acc:.2f}%  ({ph_ok}/{len(has_ph)})')
print(f'Curly parity   : {curl_acc:.2f}%  ({curl_ok}/{len(has_curl)})')
print('='*52)

In [ ]:
print('── Sample translations ──\n')
for i in range(min(10, len(test_records))):
    print(f'EN : {en_texts[i]}')
    print(f'REF: {fr_refs[i]}')
    print(f'HYP: {predictions[i]}')
    print()

## 12 · Translate New Sentences

In [ ]:
def translate(text):
    sc = {}
    norm = normalize(text, sc)
    inputs = tokenizer(
        norm, return_tensors='pt', max_length=MAX_LEN, truncation=True
    ).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            forced_bos_token_id=fra_token_id,
            num_beams=4,
            length_penalty=0.6,
            max_new_tokens=MAX_LEN,
        )
    norm_fr = tokenizer.decode(out[0], skip_special_tokens=True)
    if not validate_ph(norm, norm_fr):
        norm_fr = repair_ph(norm, norm_fr)
    return restore(norm_fr, sc)

test_sentences = [
    '{clock} Duration: 90 Mins, {skip} Skip the Line, {cancel} Free Cancellation',
    'Explore the Eiffel Tower with a skip-the-line ticket and a guided tour.',
    'Entry includes access to all three floors and the champagne bar at the summit.',
    '[Get Directions](https://goo.gl/maps/abc123)',
    '<ph x="0" type="x"><x id="x1"/></ph>Highlights<ph x="1" type="x"><x id="x2"/></ph>',
    'Photography is allowed inside the museum.',
    'Mobile tickets accepted. No printout required.',
]

print('── Translations ──\n')
for s in test_sentences:
    fr = translate(s)
    print(f'EN: {s}')
    print(f'FR: {fr}')
    print()

## 13 · Save Adapter to GCS (optional)

In [ ]:
# GCS_BUCKET = 'gs://YOUR_BUCKET/headout-nllb-adapter'
# rc = os.system(f'gsutil -m cp -r {ADAPTER_DIR} {GCS_BUCKET}/')
# if rc != 0:
#     raise RuntimeError(f'gsutil failed (exit {rc})')
# print(f'Adapter uploaded to {GCS_BUCKET}/')

## 13 · Export for Backend  \n\nMerges LoRA into the base model, saves in fp16, writes a ready-to-use `HeadoutTranslator` class, and zips everything for download.  \nBackend only needs: `pip install transformers torch sentencepiece`"

In [ ]:
import os, shutil, time, json, torch
from pathlib import Path

EXPORT_DIR = './headout-translator'
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
Path(EXPORT_DIR).mkdir()

# ── 1. Merge LoRA into base model ────────────────────────────────────────────
# merge_and_unload() dequantizes 4-bit → bf16 (our compute dtype).
# Do NOT call .to(dtype) after — bitsandbytes blocks dtype casting on quantized models.
# bf16 and fp16 are both 2 bytes/param; bf16 is preferred since the model trained in bf16.
print('Merging LoRA into base model ...')
t0 = time.time()
merged = model.merge_and_unload()
merged.save_pretrained(EXPORT_DIR, safe_serialization=True)
tokenizer.save_pretrained(EXPORT_DIR)
print(f'Merge done in {time.time()-t0:.0f}s')

# ── 2. inference_config.json ──────────────────────────────────────────────────
json.dump({
    'src_lang': SRC_LANG,
    'tgt_lang': TGT_LANG,
    'fra_token_id': int(fra_token_id),
    'max_new_tokens': MAX_LEN,
    'num_beams': 4,
    'length_penalty': 0.6,
}, open(f'{EXPORT_DIR}/inference_config.json', 'w'), indent=2)

# ── 3. translate.py — drop-in class for the backend team ─────────────────────
Path(f'{EXPORT_DIR}/translate.py').write_text('''import json, torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

class HeadoutTranslator:
    def __init__(self, model_dir: str, device: str = None):
        cfg = json.load(open(Path(model_dir) / "inference_config.json"))
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tok = AutoTokenizer.from_pretrained(model_dir)
        self.tok.src_lang = cfg["src_lang"]
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            model_dir, torch_dtype=torch.bfloat16
        ).to(self.device).eval()
        self.cfg = cfg

    def translate(self, text: str) -> str:
        inputs = self.tok(text, return_tensors="pt",
                          max_length=self.cfg["max_new_tokens"], truncation=True).to(self.device)
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                forced_bos_token_id=self.cfg["fra_token_id"],
                num_beams=self.cfg["num_beams"],
                length_penalty=self.cfg["length_penalty"],
                max_new_tokens=self.cfg["max_new_tokens"],
            )
        return self.tok.decode(out[0], skip_special_tokens=True)

    def translate_batch(self, texts: list[str]) -> list[str]:
        inputs = self.tok(texts, return_tensors="pt", padding=True,
                          max_length=self.cfg["max_new_tokens"], truncation=True).to(self.device)
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                forced_bos_token_id=self.cfg["fra_token_id"],
                num_beams=self.cfg["num_beams"],
                length_penalty=self.cfg["length_penalty"],
                max_new_tokens=self.cfg["max_new_tokens"],
            )
        return self.tok.batch_decode(out, skip_special_tokens=True)

if __name__ == "__main__":
    t = HeadoutTranslator("./headout-translator")
    print(t.translate("Skip the Line tickets for the Eiffel Tower."))
''')

# ── 4. Show export contents ───────────────────────────────────────────────────
total_mb = sum(p.stat().st_size for p in Path(EXPORT_DIR).rglob('*') if p.is_file()) / 1e6
print(f'\nExport: {EXPORT_DIR}  ({total_mb:.0f} MB)')
for p in sorted(Path(EXPORT_DIR).iterdir()):
    if p.is_file():
        print(f'  {p.name:45s} {p.stat().st_size/1e6:6.1f} MB')

# ── 5. Zip and download ───────────────────────────────────────────────────────
print('\nZipping ...')
shutil.make_archive('headout-translator', 'zip', '.', EXPORT_DIR)
print(f'headout-translator.zip  →  {Path("headout-translator.zip").stat().st_size/1e6:.0f} MB')
try:
    from google.colab import files
    files.download('headout-translator.zip')
except ImportError:
    print('Download headout-translator.zip from the Workbench file browser')

print('''
── Backend usage ─────────────────────────────────────
pip install transformers torch sentencepiece

from translate import HeadoutTranslator
t = HeadoutTranslator("./headout-translator")
t.translate("Skip the Line tickets for the Eiffel Tower.")
t.translate_batch(["text 1", "text 2"])
''')

In [ ]:
# ── Download headout-translator.zip in 500 MB chunks ─────────────────────────
# Drive mount not supported in Colab Enterprise.
# Split into parts → download each → rejoin locally with: cat part_* > headout-translator.zip
import subprocess, glob
from google.colab import files
from pathlib import Path

zip_path = 'headout-translator.zip'
assert Path(zip_path).exists(), 'Run the export cell first to create the zip'

# Clean up any old parts
subprocess.run('rm -f part_*', shell=True)

# Split into 500 MB chunks
subprocess.run(['split', '-b', '500m', zip_path, 'part_'], check=True)
parts = sorted(glob.glob('part_*'))
print(f'Split into {len(parts)} parts:')
for p in parts:
    print(f'  {p}  ({Path(p).stat().st_size/1e6:.0f} MB)')

print('\nDownloading parts one by one ...')
for p in parts:
    print(f'  ↓ {p}')
    files.download(p)

print('''
Parts downloaded. Rejoin on your local machine:
  Mac/Linux : cat part_* > headout-translator.zip && unzip headout-translator.zip
  Windows   : copy /b part_* headout-translator.zip
''')

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

INFER_MAX_LEN = 256   # inference only — longer than training MAX_LEN=64 is fine

model.eval()

txt_in  = widgets.Textarea(placeholder='Type English text here...', layout=widgets.Layout(width='100%', height='80px'))
btn     = widgets.Button(description='Translate →', button_style='primary')
out     = widgets.Output()

def on_translate(_):
    en = txt_in.value.strip()
    if not en:
        return
    with out:
        clear_output()
        print('Translating ...')
        sc = {}
        norm = normalize(en, sc)
        inputs = tokenizer(
            norm, return_tensors='pt',
            max_length=INFER_MAX_LEN, truncation=True
        ).to(DEVICE)
        n_input_tokens = inputs['input_ids'].shape[1]
        with torch.no_grad():
            ids = model.generate(
                **inputs,
                forced_bos_token_id=fra_token_id,
                num_beams=4,
                length_penalty=0.6,
                max_new_tokens=INFER_MAX_LEN,
            )
        fr_norm = tokenizer.decode(ids[0], skip_special_tokens=True)
        if not validate_ph(norm, fr_norm):
            fr_norm = repair_ph(norm, fr_norm)
        fr = restore(fr_norm, sc)
        print(f'Input tokens : {n_input_tokens}  (limit: {INFER_MAX_LEN})')
        print(f'\n🇬🇧 EN : {en}')
        print(f'🇫🇷 FR : {fr}')

btn.on_click(on_translate)
display(widgets.VBox([txt_in, btn, out]))

In [ ]:
# ── Evaluate BLEU on 100 random pairs from test set ──────────────────────────
import time, random, csv, io
import sacrebleu as scb

INFER_MAX_LEN = 256
EVAL_BS       = 16
N_SAMPLE      = 100
RANDOM_SEED   = 42

random.seed(RANDOM_SEED)
sample   = random.sample(test_records, N_SAMPLE)
en_texts = [p['en'] for p in sample]
fr_refs  = [p['fr'] for p in sample]
print(f'Sampled {N_SAMPLE} pairs from test set ({len(test_records):,} total)')

# ── Translate ─────────────────────────────────────────────────────────────────
model.eval()
predictions = []
t0 = time.time()

for i in range(0, len(en_texts), EVAL_BS):
    batch = en_texts[i:i + EVAL_BS]
    scs   = [{} for _ in batch]
    norms = [normalize(t, sc) for t, sc in zip(batch, scs)]
    inputs = tokenizer(
        norms, return_tensors='pt', padding=True,
        max_length=INFER_MAX_LEN, truncation=True,
    ).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            forced_bos_token_id=fra_token_id,
            num_beams=4,
            length_penalty=0.6,
            max_new_tokens=INFER_MAX_LEN,
        )
    decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
    for fr_norm, norm_en, sc in zip(decoded, norms, scs):
        if not validate_ph(norm_en, fr_norm):
            fr_norm = repair_ph(norm_en, fr_norm)
        predictions.append(restore(fr_norm, sc))

print(f'Translated {len(predictions)} sentences in {time.time()-t0:.0f}s')

# ── Scores ────────────────────────────────────────────────────────────────────
bleu = scb.corpus_bleu(predictions, [fr_refs])
chrf = scb.corpus_chrf(predictions, [fr_refs], word_order=2)
print(f'\n{"="*48}')
print(f'BLEU   : {bleu.score:.2f}   (target ≥ 48)')
print(f'chrF++ : {chrf.score:.2f}   (target ≥ 65)')
print(f'{"="*48}')

# ── 10 samples ────────────────────────────────────────────────────────────────
print('\n── 10 sample translations ──')
for i in range(10):
    print(f'\nEN : {en_texts[i]}')
    print(f'REF: {fr_refs[i]}')
    print(f'HYP: {predictions[i]}')

# ── Download CSV ──────────────────────────────────────────────────────────────
buf = io.StringIO()
w = csv.DictWriter(buf, fieldnames=['en', 'reference', 'hypothesis'])
w.writeheader()
for en, ref, hyp in zip(en_texts, fr_refs, predictions):
    w.writerow({'en': en, 'reference': ref, 'hypothesis': hyp})
buf.seek(0)
with open('eval_results.csv', 'w') as f:
    f.write(buf.read())

from google.colab import files as colab_files
colab_files.download('eval_results.csv')
print('eval_results.csv downloaded.')

In [ ]:
# ── DeepL vs Fine-tuned NLLB: BLEU & chrF++ on 1000 test pairs ───────────────
!pip install -q deepl

import deepl, time, random, csv, io, getpass
import sacrebleu as scb

N_SAMPLE      = 1000
RANDOM_SEED   = 42
INFER_MAX_LEN = 256
EVAL_BS       = 64   # merged model is full bf16 — more VRAM headroom than training
NLLB_BEAMS    = 2    # 2 beams: ~2× faster than 4, negligible quality drop
DEEPL_BS      = 50

# ── API key ───────────────────────────────────────────────────────────────────
DEEPL_KEY = getpass.getpass('Enter DeepL API key (hidden): ')
translator_deepl = deepl.Translator(DEEPL_KEY)

# ── Sample ────────────────────────────────────────────────────────────────────
random.seed(RANDOM_SEED)
sample   = random.sample(test_records, N_SAMPLE)
en_texts = [p['en'] for p in sample]
fr_refs  = [p['fr'] for p in sample]
print(f'Sampled {N_SAMPLE} pairs from test set')

# ── 1. DeepL translations ─────────────────────────────────────────────────────
print('\nTranslating with DeepL ...')
deepl_preds = []
t0 = time.time()
for i in range(0, len(en_texts), DEEPL_BS):
    batch = en_texts[i:i + DEEPL_BS]
    results = translator_deepl.translate_text(batch, source_lang='EN', target_lang='FR')
    deepl_preds.extend(r.text for r in results)
print(f'DeepL done in {time.time()-t0:.0f}s')

# ── 2. Fine-tuned NLLB translations ──────────────────────────────────────────
print(f'\nTranslating with fine-tuned NLLB  (batch={EVAL_BS}, beams={NLLB_BEAMS}) ...')
nllb_preds = []
model.eval()
t0 = time.time()
for i in range(0, len(en_texts), EVAL_BS):
    batch = en_texts[i:i + EVAL_BS]
    scs   = [{} for _ in batch]
    norms = [normalize(t, sc) for t, sc in zip(batch, scs)]
    inputs = tokenizer(
        norms, return_tensors='pt', padding=True,
        max_length=INFER_MAX_LEN, truncation=True,
    ).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            forced_bos_token_id=fra_token_id,
            num_beams=NLLB_BEAMS,
            length_penalty=0.6,
            max_new_tokens=INFER_MAX_LEN,
        )
    decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
    for fr_norm, norm_en, sc in zip(decoded, norms, scs):
        if not validate_ph(norm_en, fr_norm):
            fr_norm = repair_ph(norm_en, fr_norm)
        nllb_preds.append(restore(fr_norm, sc))
    print(f'  NLLB: {min(i+EVAL_BS, N_SAMPLE)}/{N_SAMPLE}  ({time.time()-t0:.0f}s)', flush=True)
print(f'NLLB done in {time.time()-t0:.0f}s')

# ── 3. Scores ─────────────────────────────────────────────────────────────────
deepl_bleu = scb.corpus_bleu(deepl_preds, [fr_refs])
deepl_chrf = scb.corpus_chrf(deepl_preds, [fr_refs], word_order=2)
nllb_bleu  = scb.corpus_bleu(nllb_preds,  [fr_refs])
nllb_chrf  = scb.corpus_chrf(nllb_preds,  [fr_refs], word_order=2)

print(f'\n{"="*52}')
print(f'{"Model":<22} {"BLEU":>8} {"chrF++":>8}')
print(f'{"-"*52}')
print(f'{"DeepL":<22} {deepl_bleu.score:>8.2f} {deepl_chrf.score:>8.2f}')
print(f'{"Fine-tuned NLLB":<22} {nllb_bleu.score:>8.2f} {nllb_chrf.score:>8.2f}')
print(f'{"="*52}')
delta_bleu = nllb_bleu.score - deepl_bleu.score
delta_chrf = nllb_chrf.score - deepl_chrf.score
print(f'{"Δ (NLLB − DeepL)":<22} {delta_bleu:>+8.2f} {delta_chrf:>+8.2f}')
print(f'{"="*52}')

# ── 4. 10 side-by-side samples ────────────────────────────────────────────────
print('\n── 10 side-by-side samples ──')
for i in range(10):
    print(f'\nEN    : {en_texts[i]}')
    print(f'REF   : {fr_refs[i]}')
    print(f'DeepL : {deepl_preds[i]}')
    print(f'NLLB  : {nllb_preds[i]}')

# ── 5. Download CSV ───────────────────────────────────────────────────────────
buf = io.StringIO()
w = csv.DictWriter(buf, fieldnames=['en', 'reference', 'deepl', 'nllb'])
w.writeheader()
for en, ref, d, n in zip(en_texts, fr_refs, deepl_preds, nllb_preds):
    w.writerow({'en': en, 'reference': ref, 'deepl': d, 'nllb': n})
buf.seek(0)
with open('deepl_vs_nllb.csv', 'w') as f:
    f.write(buf.read())
from google.colab import files as colab_files
colab_files.download('deepl_vs_nllb.csv')
print('deepl_vs_nllb.csv downloaded.')